# DocFinder — Colab Pipeline v4

**Pipeline self-healing + dashboard live + concurrence adaptative.**

## Quoi de neuf vs v3

- **Cell 3** : cloudflared tunnel #2 avec watchdog intégré — redémarre tout seul à chaque crash.
- **Cell 7** : dashboard temps réel (GPU, VRAM, RAM, CPU, I/O, tunnels, progression Qdrant, vitesse) actualisé toutes les 5 s.
- **Cell 4** : variable `DOCFINDER_HTTP_WORKERS` pour adapter la concurrence au GPU (T4 → 8, L4 → 16, A100 → 32).
- Recommandations adaptatives affichées dans le dashboard selon ressources observées.

## Pré-requis

1. Runtime GPU : Exécution → Modifier le type d'exécution → T4 / L4 / A100.
2. Tunnel #1 Mac → `docfinder.jinkohub.digital` (LaunchAgent côté Mac, vérifier avant).
3. Tunnel #2 token Colab → `encode.jinkohub.digital` (à coller cell 3).

## Ordre

1 → 2 → **(restart runtime)** → 1 → 3 → 4 → 5 → 6 → 7 → 8

Tout ce qui suit cell 2 doit tourner dans la même session après restart.

## Cell 1 — Clone / pull repo

In [ ]:
import os, subprocess

BRANCH = 'claude/review-project-cloudflare-ggcYD'
REPO = 'https://github.com/kofekod23/docfinder.git'

if not os.path.exists('/content/docfinder/.git'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO, '/content/docfinder'], check=True)
else:
    subprocess.run(['git', '-C', '/content/docfinder', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/docfinder', 'reset', '--hard', f'origin/{BRANCH}'], check=True)

head = subprocess.check_output(['git', '-C', '/content/docfinder', 'rev-parse', '--short', 'HEAD'], text=True).strip()
msg = subprocess.check_output(['git', '-C', '/content/docfinder', 'log', '-1', '--pretty=%s'], text=True).strip()
print(f'✅ HEAD → {head}: {msg}')


## Cell 2 — Installer les dépendances

⚠️ Après cette cellule : **Exécution → Redémarrer la session** puis reprendre cell 1.

In [ ]:
!pip install -q \
  FlagEmbedding==1.3.5 \
  'transformers>=4.51,<5' \
  'sentence-transformers>=3.3' \
  'fastapi>=0.110' \
  'uvicorn[standard]>=0.27' \
  httpx pydantic easyocr PyMuPDF python-docx python-pptx openpyxl \
  nvidia-ml-py psutil
print('✅ deps OK. Restart runtime si nouvelles versions.')


## Cell 3 — cloudflared + watchdog (tunnel #2 auto-restart)

Cloudflared se relance seul à chaque crash grâce au thread watchdog. Plus jamais de 530/1033.

In [ ]:
TUNNEL2_TOKEN = ''  # ← colle ici le token du 2e tunnel

import subprocess, os, time, threading

if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-qO', '/usr/local/bin/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'], check=True)
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# Kill anciens cloudflared pour repartir propre
subprocess.run(['pkill', '-f', 'cloudflared.*tunnel.*run'], check=False)
time.sleep(1)

assert TUNNEL2_TOKEN.strip(), 'TUNNEL2_TOKEN vide — colle le token du 2e tunnel'

_TUNNEL_STATE = {'restarts': 0, 'current_pid': None, 'last_restart_at': None}

def _tunnel_watchdog():
    while True:
        log = open('/content/cloudflared.log', 'ab')
        p = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', TUNNEL2_TOKEN],
            stdout=log, stderr=log)
        _TUNNEL_STATE['current_pid'] = p.pid
        _TUNNEL_STATE['last_restart_at'] = time.time()
        print(f'[tunnel-watchdog] cloudflared PID={p.pid} started', flush=True)
        p.wait()
        _TUNNEL_STATE['restarts'] += 1
        print(f'[tunnel-watchdog] cloudflared died (total restarts={_TUNNEL_STATE["restarts"]}), respawn in 5s', flush=True)
        time.sleep(5)

_wd = threading.Thread(target=_tunnel_watchdog, daemon=True, name='tunnel-wd')
_wd.start()
time.sleep(3)
print(f'✅ watchdog actif, cloudflared PID={_TUNNEL_STATE["current_pid"]}')


## Cell 4 — Variables d'environnement

### Recommandations `DOCFINDER_HTTP_WORKERS` selon le GPU

| GPU Colab   | VRAM   | HTTP workers conseillés |
|-------------|--------|-------------------------|
| T4          | 15 GB  | 8 (défaut)              |
| L4          | 22 GB  | 16                      |
| A100        | 40 GB  | 32                      |
| TPU         | —      | 4 (CPU bottleneck)      |

Le dashboard (cell 7) affiche recommandation live selon GPU util observée.

In [ ]:
import os, subprocess

# Identité
os.environ['MAC_BASE_URL']            = 'https://docfinder.jinkohub.digital'
os.environ['DOCFINDER_ROOT']          = '/Users/julien/Documents'
os.environ['COLAB_QUERY_TOKEN']       = ''  # ← même valeur que Mac .env
os.environ['CF_ACCESS_CLIENT_ID']     = ''  # ← CF Access service token ID
os.environ['CF_ACCESS_CLIENT_SECRET'] = ''  # ← CF Access service token secret

# Flags indexation
os.environ['DOCFINDER_FORCE_REINDEX']   = '0'  # '1' pour tout ré-encoder
os.environ['DOCFINDER_ENABLE_RERANKER'] = '1'  # '0' pour désactiver /rerank
os.environ['DOCFINDER_ENABLE_QWEN']     = '0'  # '1' pour charger Qwen3 (A/B Plan 2)
os.environ['DOCFINDER_COLLECTION']      = 'docfinder_v2'

# Concurrence (auto-détection GPU ensuite si laissé vide)
os.environ['DOCFINDER_HTTP_WORKERS']    = ''  # vide = auto selon GPU (cf plus bas)

# Auto-détection du GPU et des workers recommandés
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True).strip()
    gpu_name, vram = [x.strip() for x in out.split(',', 1)]
    print(f'GPU détecté : {gpu_name} ({vram})')
    if 'A100' in gpu_name:
        rec = 32
    elif 'L4' in gpu_name:
        rec = 16
    elif 'T4' in gpu_name:
        rec = 8
    else:
        rec = 8
    if not os.environ['DOCFINDER_HTTP_WORKERS']:
        os.environ['DOCFINDER_HTTP_WORKERS'] = str(rec)
    print(f'→ DOCFINDER_HTTP_WORKERS = {os.environ["DOCFINDER_HTTP_WORKERS"]} (recommandé pour ce GPU)')
except Exception as e:
    print(f'[warn] GPU détection impossible : {e}')
    os.environ['DOCFINDER_HTTP_WORKERS'] = os.environ['DOCFINDER_HTTP_WORKERS'] or '8'

for k in ('COLAB_QUERY_TOKEN', 'CF_ACCESS_CLIENT_ID', 'CF_ACCESS_CLIENT_SECRET'):
    assert os.environ[k].strip(), f'{k} vide'
print(f'\n✅ env OK. Flags : FORCE={os.environ["DOCFINDER_FORCE_REINDEX"]} RERANK={os.environ["DOCFINDER_ENABLE_RERANKER"]} QWEN={os.environ["DOCFINDER_ENABLE_QWEN"]} WORKERS={os.environ["DOCFINDER_HTTP_WORKERS"]}')


## Cell 5 — Anti-idle navigateur (JS à paster en DevTools)

Empêche Colab de déconnecter le runtime après 90 min d'onglet inactif.

**F12 → Console** → paste ceci :

```javascript
function ClickConnect(){
    var btn = document.querySelector('colab-connect-button');
    if (btn) btn.click();
    console.log('keepalive ' + new Date().toLocaleTimeString());
}
setInterval(ClickConnect, 60000);
```

## Cell 6 — Smoke test Mac

In [ ]:
import httpx, os

headers = {
    'CF-Access-Client-Id': os.environ['CF_ACCESS_CLIENT_ID'],
    'CF-Access-Client-Secret': os.environ['CF_ACCESS_CLIENT_SECRET'],
    'User-Agent': 'DocFinder/1.0',
}
with httpx.Client(headers=headers, timeout=15) as c:
    for path in ('/health', '/files/list?root=/Users/julien/Documents&limit=1', '/admin/progress'):
        try:
            r = c.get(os.environ['MAC_BASE_URL'] + path)
            print(f'{path:50} → {r.status_code}')
        except Exception as e:
            print(f'{path:50} → ERR {type(e).__name__}')
print('\n(si /health != 200 → Mac uvicorn down ou tunnel #1 mort)')


## Cell 7 — Dashboard temps réel (GPU / VRAM / RAM / Tunnels / Indexation)

Dashboard live qui s'auto-actualise toutes les 5 s jusqu'à interruption.

**Lance cette cellule APRÈS la cell 8** (pipeline) — le dashboard monitore le pipeline.

In [ ]:
import time, subprocess, os, threading, httpx
import psutil
from IPython.display import display, clear_output, HTML
from datetime import datetime

_DASH_INTERVAL_S = 5
_DASH_STATE = {'prev_points': None, 'prev_ts': None, 'rates': []}

_headers = {
    'CF-Access-Client-Id': os.environ['CF_ACCESS_CLIENT_ID'],
    'CF-Access-Client-Secret': os.environ['CF_ACCESS_CLIENT_SECRET'],
    'User-Agent': 'DocFinder/1.0',
}

def _probe_gpu():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw',
             '--format=csv,noheader,nounits'], text=True).strip().split(',')
        name, util, mem_u, mem_t, temp, pw = [x.strip() for x in out]
        return {'name': name, 'util_pct': int(util), 'vram_used_mb': int(mem_u),
                'vram_total_mb': int(mem_t), 'temp_c': int(temp), 'power_w': float(pw)}
    except Exception as e:
        return {'error': str(e)}

def _probe_tunnels():
    r = {'mac': None, 'colab_encode': None}
    try:
        r['mac'] = httpx.get(os.environ['MAC_BASE_URL'] + '/health', headers=_headers, timeout=5).status_code
    except Exception: r['mac'] = -1
    # Tunnel #2 self-check via localhost (pas via Cloudflare pour éviter hairpin)
    try:
        r['colab_encode'] = httpx.get('http://127.0.0.1:8001/healthz', timeout=3).status_code
    except Exception: r['colab_encode'] = -1
    return r

def _probe_pipeline():
    try:
        r = httpx.get(os.environ['MAC_BASE_URL'] + '/admin/progress', headers=_headers, timeout=5)
        if r.status_code == 200:
            return r.json()
    except Exception: pass
    return {}

def _probe_qdrant():
    try:
        r = httpx.get(os.environ['MAC_BASE_URL'] + '/admin/db?v=2', headers=_headers, timeout=5)
        if r.status_code == 200:
            # Extract points count from HTML if needed, else use a different endpoint
            pass
    except Exception: pass
    return None

def _bar(pct, width=20):
    pct = max(0, min(100, int(pct)))
    fill = int(pct * width / 100)
    return '█' * fill + '░' * (width - fill)

def _color(pct, good_low=False):
    if good_low:
        return '#0a0' if pct < 50 else ('#a80' if pct < 80 else '#c00')
    return '#0a0' if pct > 80 else ('#a80' if pct > 40 else '#c00')

def _recommend(gpu, http_workers, rate_per_min):
    recs = []
    if 'util_pct' in gpu:
        util = gpu['util_pct']
        if util < 40 and rate_per_min > 1:
            recs.append(f'⬆️ GPU à {util}% seulement — peut augmenter DOCFINDER_HTTP_WORKERS ({http_workers} → {http_workers*2})')
        elif util > 90:
            recs.append(f'⬇️ GPU à {util}% — réduire DOCFINDER_HTTP_WORKERS ({http_workers} → max({http_workers//2},2))')
        if 'vram_used_mb' in gpu and gpu['vram_total_mb']:
            vram_pct = gpu['vram_used_mb'] * 100 / gpu['vram_total_mb']
            if vram_pct > 90:
                recs.append(f'⚠️ VRAM {vram_pct:.0f}% — risque OOM')
    return recs

def _render():
    now = time.time()
    gpu = _probe_gpu()
    tun = _probe_tunnels()
    pipe = _probe_pipeline()
    mem = psutil.virtual_memory()
    cpu = psutil.cpu_percent(interval=None)
    loadavg = os.getloadavg()
    http_workers = int(os.environ.get('DOCFINDER_HTTP_WORKERS', '8'))

    # Calcul vitesse
    points_now = pipe.get('done', 0)
    rate_per_min = 0
    if _DASH_STATE['prev_points'] is not None:
        dt = now - _DASH_STATE['prev_ts']
        if dt > 0:
            rate_per_min = (points_now - _DASH_STATE['prev_points']) * 60 / dt
            _DASH_STATE['rates'].append(rate_per_min)
            _DASH_STATE['rates'] = _DASH_STATE['rates'][-12:]  # garder 1 min glissante
    _DASH_STATE['prev_points'] = points_now
    _DASH_STATE['prev_ts'] = now
    avg_rate = sum(_DASH_STATE['rates']) / len(_DASH_STATE['rates']) if _DASH_STATE['rates'] else 0

    # Tunnel status
    def _status_chip(val):
        if val == 200: return '🟢 200'
        if val == -1: return '🔴 DOWN'
        return f'🟡 {val}'

    # Progress bar
    total = pipe.get('total', 0) or 1
    pct = pipe.get('done', 0) * 100 / total
    eta_s = pipe.get('eta_seconds', 0) or 0
    eta_fmt = f'{eta_s//60:.0f}m{eta_s%60:02.0f}s' if eta_s else 'calcul…'

    # GPU bar
    gpu_util = gpu.get('util_pct', 0)
    vram_pct = (gpu.get('vram_used_mb', 0) * 100 / gpu.get('vram_total_mb', 1)) if gpu.get('vram_total_mb') else 0

    recs = _recommend(gpu, http_workers, avg_rate)

    html = f'''
<style>
.dash {{ font-family: monospace; font-size: 13px; line-height: 1.4; }}
.dash h3 {{ margin: 8px 0 4px; border-bottom: 1px solid #ccc; }}
.bar {{ font-family: monospace; }}
.chip {{ display: inline-block; padding: 2px 6px; border-radius: 3px; margin-right: 6px; }}
.ok {{ background: #d4edda; }}
.warn {{ background: #fff3cd; }}
.bad {{ background: #f8d7da; }}
table.dash-tbl {{ border-collapse: collapse; }}
table.dash-tbl td {{ padding: 2px 10px 2px 0; }}
</style>
<div class="dash">
<h3>🚀 DocFinder v4 — {datetime.now().strftime("%H:%M:%S")}</h3>

<h3>Tunnels</h3>
<table class="dash-tbl">
<tr><td>Mac (docfinder.jinkohub.digital)</td><td>{_status_chip(tun["mac"])}</td></tr>
<tr><td>Colab encode (127.0.0.1:8001)</td><td>{_status_chip(tun["colab_encode"])}</td></tr>
<tr><td>cloudflared restarts</td><td>{_TUNNEL_STATE.get("restarts", 0)}</td></tr>
</table>

<h3>Indexation</h3>
<table class="dash-tbl">
<tr><td>Progression</td><td>{pipe.get("done", 0)}/{pipe.get("total", "?")} ({pct:.1f}%)</td></tr>
<tr><td>Courant</td><td>{pipe.get("current_doc", "idle")[:50]}</td></tr>
<tr><td>Failed</td><td>{pipe.get("failed", 0)}</td></tr>
<tr><td>Vitesse (docs/min)</td><td>{avg_rate:.1f} (instant {rate_per_min:.1f})</td></tr>
<tr><td>ETA</td><td>{eta_fmt}</td></tr>
</table>
<div class="bar">[{_bar(pct)}] {pct:.1f}%</div>

<h3>GPU</h3>
<table class="dash-tbl">
<tr><td>Modèle</td><td>{gpu.get("name", "?")}</td></tr>
<tr><td>Util</td><td>{gpu_util}% [{_bar(gpu_util)}]</td></tr>
<tr><td>VRAM</td><td>{gpu.get("vram_used_mb", 0)}/{gpu.get("vram_total_mb", 0)} MB ({vram_pct:.0f}%) [{_bar(vram_pct)}]</td></tr>
<tr><td>Temp / Power</td><td>{gpu.get("temp_c", "?")}°C / {gpu.get("power_w", 0):.1f}W</td></tr>
</table>

<h3>Système Colab</h3>
<table class="dash-tbl">
<tr><td>CPU</td><td>{cpu:.0f}% [{_bar(cpu)}]</td></tr>
<tr><td>RAM</td><td>{mem.used/1024**3:.1f}/{mem.total/1024**3:.1f} GB ({mem.percent:.0f}%) [{_bar(mem.percent)}]</td></tr>
<tr><td>Load (1/5/15 min)</td><td>{loadavg[0]:.1f} / {loadavg[1]:.1f} / {loadavg[2]:.1f}</td></tr>
<tr><td>HTTP workers</td><td>{http_workers}</td></tr>
</table>

<h3>Recommandations adaptatives</h3>
<div>{"<br>".join(recs) if recs else "✅ config OK — aucun ajustement recommandé"}</div>
</div>
'''
    clear_output(wait=True)
    display(HTML(html))

print('Dashboard démarré. Ctrl+M+I (interrupt) pour arrêter.')
try:
    while True:
        _render()
        time.sleep(_DASH_INTERVAL_S)
except KeyboardInterrupt:
    print('Dashboard arrêté.')


## Cell 8 — Lancement unifié (query_server + pipeline self-healing)

**À lancer AVANT la cell 7** (qui le monitore).

Cellule bloquante — tant qu'elle tourne tout est live. Stop cellule = arrêt propre.

In [ ]:
exec(open('/content/docfinder/colab_helpers_cell.py').read())


## Workflow recommandé

1. Cell 1-4 séquentiellement (avec restart runtime après cell 2).
2. **Cell 8** dans un onglet (bloquante, logs pipeline).
3. **Cell 7** dans un autre onglet du même notebook (dashboard, bloquante aussi).
4. Paste JS anti-idle (cell 5) en console navigateur.
5. Fermer l'ordinateur. Dashboard + pipeline tournent seuls. cloudflared se relance seul via watchdog.

## Ajuster la cadence à la volée

Si le dashboard recommande `⬆️ GPU à X% seulement — augmenter workers` :

1. Interrompre la cell 8 (Stop).
2. Revenir cell 4, mettre la nouvelle valeur (ex. `DOCFINDER_HTTP_WORKERS='16'`).
3. Relancer cell 8.

Le checkpoint `/content/checkpoint_v2.json` reprend où on était, pas de re-travail.

## Résumé des cellules

| # | Rôle                                   | Dépend de  |
|---|----------------------------------------|------------|
| 1 | git pull repo                          | —          |
| 2 | pip install (restart runtime après)    | 1          |
| 3 | cloudflared tunnel #2 + watchdog       | 2          |
| 4 | env vars + auto-détection GPU workers  | —          |
| 5 | (markdown) JS anti-idle                | —          |
| 6 | smoke test Mac                         | 4          |
| 7 | dashboard temps réel (bloquant)        | 4, 8       |
| 8 | pipeline self-healing (bloquant)       | 3, 4       |